# Nepali Legal QA — Colab Backend

**Runtime: GPU (T4)** — Runtime → Change runtime type → GPU

This notebook runs the RAG backend (FastAPI + SLM) on Colab. The forum backend runs locally for the realtime demo.

Run all cells top to bottom. You will be prompted to upload `augmented_nepali_legal_rag.txt` in Cell 2.

## Cell 1 — Install dependencies & clone repo

## Cell 0 — Repo setup (public)

This repo is public, so no GitHub token is required.

If you want to use a fork, update `REPO_URL` in Cell 1.

## Optional — Colab Secrets for keys

Add these in Colab Secrets (left sidebar 🔑) before running the next cells:

- `HF_TOKEN`
- `GROQ_API_KEY`, `GROQ_API_KEY_2`, `GROQ_API_KEY_3`, `GROQ_API_KEY_4`
- `NGROK_TOKEN`
- `GOOGLE_CLIENT_ID` (optional for auth)
- `SECRET_KEY` (optional for auth)

This keeps secrets out of the notebook.

In [ ]:
import os
import subprocess

REPO_URL = "https://github.com/dipsankadariya/RAG-based-QA-System.git"
BRANCH = "dipsan-forum"
REPO_DIR = "/content/RAG-based-QA-System"
BACKEND_DIR = os.path.join(REPO_DIR, "nepali-legal-qa", "backend")

if not os.path.exists(REPO_DIR):
    print("Cloning repo...")
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    print("Repo already cloned. Updating...")
    os.chdir(REPO_DIR)
    subprocess.run(["git", "fetch", "origin"], check=True)
    subprocess.run(["git", "checkout", BRANCH], check=True)
    subprocess.run(["git", "pull", "origin", BRANCH], check=True)

os.chdir(BACKEND_DIR)
print(f"Working directory: {os.getcwd()}")

print("Installing dependencies...")
subprocess.run(["pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run(["pip", "install", "-q", "pyngrok"], check=True)
print("Done.")

In [ ]:
from google.colab import files
import os

print('Uploading augmented_nepali_legal_rag.txt...')
uploaded = files.upload()

# Move uploaded file to backend directory
for filename in uploaded.keys():
    if filename == 'augmented_nepali_legal_rag.txt':
        os.rename(filename, f'{os.getcwd()}/augmented_nepali_legal_rag.txt')
        print(f'✓ File uploaded and ready at: {os.getcwd()}/augmented_nepali_legal_rag.txt')
    else:
        print(f'Warning: Expected augmented_nepali_legal_rag.txt but got {filename}')

## Cell 2 — Upload augmented_nepali_legal_rag.txt file

## Cell 3 — HuggingFace login

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise ValueError("HF_TOKEN is missing. Add it in Colab Secrets and retry.")

login(token=hf_token, add_to_git_credential=False)
print("HuggingFace login OK")

## Cell 4 — Set Groq API keys

In [ ]:
import os
from google.colab import userdata

keys = [
    "GROQ_API_KEY",
    "GROQ_API_KEY_2",
    "GROQ_API_KEY_3",
    "GROQ_API_KEY_4",
]
missing = [key for key in keys if not userdata.get(key)]

if missing:
    raise ValueError(f"Missing secrets: {', '.join(missing)}")

for key in keys:
    os.environ[key] = userdata.get(key)

print("Groq API keys set")

In [ ]:
# Install Google authentication packages
!pip install google-auth-oauthlib google-auth python-jose PyJWT

import os
from google.colab import userdata

# Colab Secrets can raise if missing; handle gracefully.
try:
    google_client_id = userdata.get("GOOGLE_CLIENT_ID")
except Exception:
    google_client_id = None

try:
    secret_key = userdata.get("SECRET_KEY")
except Exception:
    secret_key = None

os.environ["GOOGLE_CLIENT_ID"] = google_client_id or "925963866872-ufg3vgsrj8tia84r0q58hbqrht1g1sb0.apps.googleusercontent.com"
os.environ["SECRET_KEY"] = secret_key or "your-secret-key-change-in-production"

if not os.environ["GOOGLE_CLIENT_ID"]:
    print("GOOGLE_CLIENT_ID not set (login will not work).")
else:
    print("GOOGLE_CLIENT_ID set.")

print("SECRET_KEY configured.")

## Cell 4.5 — Install & Configure Google Authentication

## Cell 5 — Start ngrok tunnel

Copy the printed URL into `frontend/.env` as `VITE_API_BASE=...` then restart `npm run dev`.

The forum API runs locally on your machine (`http://localhost:8001`) for the realtime demo.

In [ ]:
from google.colab import userdata
from pyngrok import ngrok

# Colab Secrets can raise if missing; handle gracefully.
try:
    ngrok_token = userdata.get("NGROK_TOKEN")
except Exception:
    ngrok_token = None

ngrok_token = ngrok_token or "3BWKR1ZKqXMZGmiDFGef66UW34d_5MYW6PYcg888saxnn7Gbd"
if not ngrok_token:
    raise ValueError("NGROK_TOKEN is missing. Add it in Colab Secrets and retry.")

ngrok.set_auth_token(ngrok_token)
tunnel = ngrok.connect(8000, "http")
public_url = tunnel.public_url

print("=" * 60)
print("  BACKEND PUBLIC URL")
print("=" * 60)
print(f"  {public_url}")
print()
print("  Paste this into frontend/.env on your local machine:")
print(f"  VITE_API_BASE={public_url}")
print("=" * 60)

In [ ]:
import os

# Set the file path for Colab (file is in current backend directory)
os.environ['DOC_FILE_PATH'] = './augmented_nepali_legal_rag.txt'

# Verify file exists
if os.path.exists(os.environ['DOC_FILE_PATH']):
    print(f'✓ Document file found: {os.environ["DOC_FILE_PATH"]}')
else:
    raise RuntimeError(f'Document file not found at: {os.environ["DOC_FILE_PATH"]}')

## Cell 6 — Start the FastAPI server

Keep this cell running while using the app.
Startup takes **2–3 min** (SLM download + FAISS build from your .txt file).
Wait for: `Application startup complete.`

In [ ]:
!uvicorn main:app --host 0.0.0.0 --port 8000 --workers 1 --timeout-keep-alive 120

---
## Quick tests (new cell, after server starts)

```python
import requests

# Test health endpoint
print("Health check:")
print(requests.get(public_url + '/api/health').json())
print()

# Test dual answer endpoint (HyDE vs Baseline retrieval)
print("Testing dual-answer query:")
r = requests.post(public_url + '/api/query',
    json={'question': 'नेपालमा सम्बन्ध विच्छेद कसरी गर्ने?'})
result = r.json()
print(f"Processing time: {result['processing_time']}s\n")
print(f"[BASELINE RETRIEVAL]\n{result['baseline_answer']}\n")
print(f"[HYDE RETRIEVAL]\n{result['hyde_answer']}\n")
print(f"[HYDE PASSAGE GENERATED]\n{result['hyde_passage']}")
```

**Updated API Response Fields:**
- `baseline_answer` — Answer using traditional retrieval
- `hyde_answer` — Answer using HyDE-generated passage retrieval
- `baseline_answer_in_english` — Baseline answer translated to English
- `hyde_answer_in_english` — HyDE answer translated to English
- `baseline_retrieved_docs` — Top-k documents for baseline
- `hyde_retrieved_docs` — Top-k documents for HyDE retrieval
- `hyde_passage` — The generated hypothetical document for HyDE
- `processing_time` — Total query processing time